# GPU correction run on Kaggle (revision, 40 pages)

Runs every correction model on all 40 evaluation pages, both OCR engines, plus the
clean-text control (ground truth as input), with plain greedy decoding. See
`context/decisions.md` D30 and D31.

**Before running:**
1. Settings: phone-verified account (needed for GPU and internet).
2. Add data: the dataset holding `ocr_eval40.jsonl`. To resume a stopped run, also add
   the previous run's output (it holds a partial `gpu_eval40.jsonl`).
3. Session options: Accelerator = GPU T4 x2 (or P100), Internet = on.
4. Add-ons, Secrets: `HF_TOKEN` (Hugging Face read token; Gemma and Llama are gated).
5. Save Version, then "Save & Run All (Commit)". It runs in the background; the browser can be closed.

When it finishes, download `gpu_eval40.jsonl` from the version's Output tab.

In [ ]:
import glob, os, shutil, subprocess, time

# Kaggle background runs stop at 12 h; start no new page after 11 h.
DEADLINE = time.time() + 11 * 3600

REPO_DIR = "/tmp/bengali-postocr-sllm"
OUT = "/kaggle/working/gpu_eval40.jsonl"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/Ariffurrhmn/bengali-postocr-sllm.git", REPO_DIR], check=True)
subprocess.run("pip install -q -U transformers accelerate sentencepiece jiwer", shell=True, check=True)
subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"])
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
# Inputs: the 40-page OCR file, and (optional) a partial output from an earlier run.
ocr_files = glob.glob("/kaggle/input/**/ocr_eval40.jsonl", recursive=True)
assert ocr_files, "Attach the dataset that contains ocr_eval40.jsonl"
OCR = ocr_files[0]
print("OCR file:", OCR, "pages:", sum(1 for _ in open(OCR, encoding="utf-8")))

previous = glob.glob("/kaggle/input/**/gpu_eval40.jsonl", recursive=True)
if previous and not os.path.exists(OUT):
    shutil.copy(previous[0], OUT)
    print("Resuming from", previous[0])
print("Already done:", sum(1 for _ in open(OUT, encoding="utf-8")) if os.path.exists(OUT) else 0)

In [ ]:
def sweep(models, engines):
    cmd = ["python", "run_sweep.py", "--split", "eval", "--models", models,
           "--engines", engines, "--approaches", "chunked", "--decoding", "plain",
           "--ocr-path", OCR, "--out", OUT,
           "--stop-after-minutes", str(max(0, (DEADLINE - time.time()) / 60))]
    # Progress goes to the log; only the summary lines are kept in the notebook output.
    with open("/kaggle/working/sweep.log", "a", encoding="utf-8") as log:
        subprocess.run(cmd, cwd=f"{REPO_DIR}/correction", stdout=log, stderr=log)
    subprocess.run(r"grep -E '^\[|FAILED|Error' /kaggle/working/sweep.log | tail -5", shell=True)

# Priority order: each step is useful on its own if the session runs out.
sweep("gemma-2b,qwen3-1.7b,llama3.2-1b,titullm-1b,banglat5", "tesseract,easyocr")

In [ ]:
sweep("phi4-mini,titullm-1b-instruct,titullm-3b-instruct", "tesseract,easyocr")

In [ ]:
sweep("gemma-2b,qwen3-1.7b,llama3.2-1b,titullm-1b,banglat5,phi4-mini,titullm-1b-instruct,titullm-3b-instruct",
      "ground_truth")

In [ ]:
import collections, json
rows = [json.loads(l) for l in open(OUT, encoding="utf-8") if l.strip()]
count = collections.Counter((r["model_key"], r["engine"]) for r in rows)
for key in sorted(count):
    print(key, count[key])
print("total", len(rows), "of", 8 * 3 * 40, "| errors:", sum(1 for r in rows if r.get("error")))